# Fase 0 - Exploración del Dataset IroSvA
Análisis estadístico descriptivo de las tres variantes dialectales del corpus IroSvA (Ortega-Bueno et al., 2019): distribución de clases y presencia de variables lingüísticas relevantes para detección de ironía.

## 1. Importación de librerías

In [7]:
import pandas as pd
import re
import emoji

## 2. Carga de las tres variantes dialectales

In [8]:
df_mx = pd.read_csv('../data/irosva.mx.training.csv')
df_es = pd.read_csv('../data/irosva.es.training.csv')
df_cu = pd.read_csv('../data/irosva.cu.training.csv')

print(f"Columnas: {df_mx.columns.tolist()}")
print(f"Tipos:    {df_mx.dtypes.to_dict()}")
print(df_mx.head(3))

Columnas: ['ID', 'TOPIC', 'IS_IRONIC', 'MESSAGE']
Tipos:    {'ID': dtype('O'), 'TOPIC': dtype('O'), 'IS_IRONIC': dtype('int64'), 'MESSAGE': dtype('O')}
                                 ID           TOPIC  IS_IRONIC  \
0  6424ee0864a0af40660686e135f5652b  asuntosConacyt          1   
1  f59978451dd7fb228830fed2ae00c3ef  asuntosConacyt          1   
2  280963c5eb0d162858caf3480a7ea08c  asuntosConacyt          1   

                                             MESSAGE  
0  Rica económicamente, pero muy pobre en objetiv...  
1  En algo tiene razón, mafias hay en todo, hasta...  
2  ¿De cuándo acá tan preocupados por la ciencia ...  


## 3. Validación de calidad: nulos y duplicados

In [9]:
for nombre, df in [('México', df_mx), ('España', df_es), ('Cuba', df_cu)]:
    nulos = df.isnull().sum().sum()
    dups  = df.duplicated(subset='MESSAGE').sum()
    print(f"{nombre}: {nulos} nulos | {dups} duplicados en MESSAGE")

México: 0 nulos | 1 duplicados en MESSAGE
España: 0 nulos | 2 duplicados en MESSAGE
Cuba: 0 nulos | 0 duplicados en MESSAGE


## 4. Distribución de clases por variante
El dataset presenta un desbalance 2:1 (no irónico:irónico) consistente en las tres variantes (Ortega-Bueno et al., 2019).

In [10]:
for nombre, df in [('México', df_mx), ('España', df_es), ('Cuba', df_cu)]:
    total      = len(df)
    ironico    = df['IS_IRONIC'].sum()
    no_ironico = total - ironico
    print(f"{nombre}: Total={total} | Irónico={ironico} ({ironico/total*100:.1f}%) | No irónico={no_ironico} ({no_ironico/total*100:.1f}%)")

México: Total=2400 | Irónico=800 (33.3%) | No irónico=1600 (66.7%)
España: Total=2400 | Irónico=800 (33.3%) | No irónico=1600 (66.7%)
Cuba: Total=2400 | Irónico=800 (33.3%) | No irónico=1600 (66.7%)


## 5. Ejemplos de tweets irónicos y no irónicos

In [11]:
print("=== TWEETS IRÓNICOS ===")
for tweet in df_mx[df_mx['IS_IRONIC']==1]['MESSAGE'].head(5).values:
    print("-", tweet)

print("\n=== TWEETS NO IRÓNICOS ===")
for tweet in df_mx[df_mx['IS_IRONIC']==0]['MESSAGE'].head(5).values:
    print("-", tweet)

=== TWEETS IRÓNICOS ===
- Rica económicamente, pero muy pobre en objetividad.
- En algo tiene razón, mafias hay en todo, hasta en su 4t.
- ¿De cuándo acá tan preocupados por la ciencia y la investigación?
- De una vez que paren las titulaciones, que todos sean pasantes, así todos tendrán el mismo nivel que el director de Pemex y el subdirector de Conacyt 😡
- @LopesDorigaa @Dolores_PL  Es el que también te atiende tu tiendita.. a la Padierna...!!

=== TWEETS NO IRÓNICOS ===
- Lo más grave es que en la entrevista menciona que, el modelo científico que debemos seguir es el de Cuba. ¿De dónde salió ésta gente? Su visión del mundo es completamente anacrónica y desinformada.
- Y tanta gente capacitada que se fue a  la banca por causa de estos ineptos
- De nada; si se van a gastar mis impuestos que mejor que sea en algo que nos mejore como país.
- Ellos quieren juventudes atrapadas en la ignorancia, que no salgan y vean lo que pasa en el mundo, que no aprendan. El dominio de AMLO SOBRE LOS CI

## 6. Análisis de variables lingüísticas por clase
Se analizan 7 variables lingüísticas asociadas a la detección de sarcasmo en la literatura: exclamaciones, interrogaciones, mayúsculas enfáticas, emojis, risas, negaciones e incongruencia sentimental (González-Ibáñez et al., 2011; Joshi et al., 2015; Ortega-Bueno et al., 2022). Se extraen del texto original antes del preprocesamiento para capturar emojis y patrones en su forma natural.

In [12]:
# Lexicons simples para incongruencia sentimental
NEGACIONES = {'no','nunca','jamás','ni','tampoco','nada','nadie','ningún','sin'}

def analizar_features(texto):
    texto_str  = str(texto)
    texto_lower = texto_str.lower()
    palabras   = re.findall(r'\b\w+\b', texto_lower)
    return {
        'exclamaciones':   texto_str.count('!') > 0,
        'interrogaciones': texto_str.count('?') > 0,
        'mayusculas':      bool(re.search(r'\b[A-ZÁÉÍÓÚÜÑ]{3,}\b', texto_str)),
        'emojis':          any(c in emoji.EMOJI_DATA for c in texto_str),
        'risas':           bool(re.search(r'\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xs+|xd+|lol+|jsjs)\b', texto_lower)),
        'negaciones':      sum(1 for p in palabras if p in NEGACIONES) > 0
    }

df_all = pd.concat([df_mx, df_es, df_cu], ignore_index=True)

features_df = df_all['MESSAGE'].apply(analizar_features).apply(pd.Series)
df_all = pd.concat([df_all, features_df], axis=1)

print("Presencia de variables lingüísticas por clase (%):\n")
for feature in ['exclamaciones','interrogaciones','mayusculas','emojis','risas','negaciones']:
    por_clase = df_all.groupby('IS_IRONIC')[feature].mean() * 100
    print(f"{feature:20} No irónico: {por_clase[0]:.1f}% | Irónico: {por_clase[1]:.1f}%")

Presencia de variables lingüísticas por clase (%):

exclamaciones        No irónico: 9.7% | Irónico: 17.2%
interrogaciones      No irónico: 19.9% | Irónico: 23.5%
mayusculas           No irónico: 17.6% | Irónico: 19.1%
emojis               No irónico: 7.3% | Irónico: 9.8%
risas                No irónico: 0.4% | Irónico: 0.9%
negaciones           No irónico: 44.5% | Irónico: 37.8%
